# Generation, oversampling, and traceability

Generate synthetic rows and inspect their trace records.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from mimic import MIMIC, GenerationPolicy, ResNetEncoder, ForestConditionalSampler

rng = np.random.default_rng(2)

def make_spiral(label, n, phase, noise=0.12):
    theta = np.linspace(0.45, 4.8 * np.pi, n)
    radius = np.linspace(0.25, 4.0, n)
    x = radius * np.cos(theta + phase) + rng.normal(0, noise, n)
    y = radius * np.sin(theta + phase) + rng.normal(0, noise, n)
    return pd.DataFrame({"x": x, "y": y, "label": label})

major = make_spiral("majority", 180, phase=0.0)
minor_full = make_spiral("minority", 180, phase=np.pi)
minor = minor_full.sample(n=90, random_state=2).sort_index().reset_index(drop=True)

df = pd.concat([major, minor], ignore_index=True)
df["id"] = np.arange(len(df))

display(df["label"].value_counts().rename_axis("label").to_frame("count").style.set_caption("Original class balance"))
display(df.head(8).style.set_caption("Original two-spiral training rows"))


In [ ]:
mimic = MIMIC(
    ignore_columns=["id"],
    regression_columns=["x", "y"],
    classification_columns=["label"],
    encoder=ResNetEncoder(embedding_dim=16, hidden_dim=64, n_layers=4, dropout=0.05, max_epochs=30, patience=5, batch_size=64, random_state=2),
    decoder=ForestConditionalSampler(n_estimators=120, random_state=2),
    policy=GenerationPolicy(method="displacement", neighbour_mode="normal", n_neighbors=5, lambda_range=(0.0, 0.25)),
    n_bootstrap=1,
    random_state=2,
)
mimic.fit(df)
synthetic, trace = mimic.sample(12, return_trace=True)

display(synthetic.head(8).style.set_caption("Displacement-generated sample rows"))
display(trace.head(8).style.set_caption("Displacement generation trace"))


In [ ]:
minority_needed = df["label"].value_counts()["majority"] - df["label"].value_counts()["minority"]
minority_synthetic, minority_trace = mimic.sample(
    minority_needed,
    condition={"label": "minority"},
    return_trace=True,
)
balanced = pd.concat([df.drop(columns=["id"]), minority_synthetic], ignore_index=True)

display(balanced["label"].value_counts().rename_axis("label").to_frame("count").style.set_caption("Class balance after minority-conditioned displacement oversampling"))
display(minority_synthetic.head(8).style.set_caption("Synthetic minority spiral rows"))
display(minority_trace.head(8).style.set_caption("Minority-conditioned oversampling trace"))


In [ ]:
plot_df = pd.concat(
    [
        df.drop(columns=["id"]).assign(source="original"),
        minority_synthetic.assign(source="generated"),
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(6, 5))
for (source, label), part in plot_df.groupby(["source", "label"]):
    if source == "original" and label == "majority":
        color, marker, alpha, size = "#9aa0a6", "o", 0.42, 28
    elif source == "original" and label == "minority":
        color, marker, alpha, size = "#1f77b4", "o", 0.9, 46
    else:
        color, marker, alpha, size = "#ff7f0e", "x", 0.9, 52
    ax.scatter(part["x"], part["y"], label=f"{source} {label}", color=color, marker=marker, alpha=alpha, s=size)
ax.set_title("Displacement oversampling on undersampled two spirals")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.legend(frameon=False)
fig.tight_layout()
